# Notebook 03e — Refit Hybrid Calibrators (save as objects)

**Purpose:** The original calibration notebooks (`03_*_v2.ipynb`) saved only the calibrated probability arrays, not the fitted calibrator objects. This blocks any downstream analysis that needs to apply calibration to new inputs (e.g., SHAP on calibrated outputs, alert generation in deployment simulation).

**What this notebook does:**
1. For each of 18 5-class models: load raw calib + test probabilities
2. Refit per-class hybrid calibrators using IDENTICAL logic to original notebook (`fit_calibrator` from Notebook 03 helpers)
3. Save fitted calibrators as `.joblib` files per model
4. **Verify:** apply refitted calibrators to test set, compare against saved `*_test_proba_calibrated.npy` arrays. Must match to within `atol=1e-10`.

**Hybrid strategy (PLATT_THRESHOLD=30, verified from saved strategies):**
- Isotonic regression with `out_of_bounds='clip'` if n_calib >= 30
- Platt scaling (`LogisticRegression(C=1e10, solver='lbfgs')`) if n_calib < 30
- Row renormalisation to sum to 1

**Time estimate:** ~5-15 min total.

**Output:** `calibrators/{dataset}_v2/{model_name}_hybrid_fitted.joblib`

**Stop condition:** If verification fails for any model, script halts with diagnostic. Do not proceed to Notebook 04b until all 18 models verify cleanly.

## 1. Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, shutil
REPO = '/content/drive/MyDrive/XIDS_Research/xids-research'
os.chdir(REPO)

for f in ['.gitconfig', '.git-credentials']:
    src = f'/content/drive/MyDrive/XIDS_Research/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'/root/{f}')
        if f == '.git-credentials':
            os.chmod(f'/root/{f}', 0o600)

print(f'✓ Ready in: {os.getcwd()}')

Mounted at /content/drive
✓ Ready in: /content/drive/MyDrive/XIDS_Research/xids-research


In [2]:
import numpy as np
import joblib
import json
from pathlib import Path
from collections import Counter
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss
import warnings
warnings.filterwarnings('ignore')

PLATT_THRESHOLD = 30
DATASETS = ['nsl_kdd_v2', 'unsw_nb15_v2', 'cic_ids2017_v2']
MODELS_5CLASS = ['rf_5class_smote', 'xgb_5class_smote', 'dnn_5class_smote',
                 'rf_5class_cw', 'xgb_5class_cw', 'dnn_5class_cw']

print(f'PLATT_THRESHOLD = {PLATT_THRESHOLD}')
print(f'Datasets: {DATASETS}')
print(f'Models to refit per dataset: {MODELS_5CLASS}')
print(f'Total: {len(DATASETS) * len(MODELS_5CLASS)} = {len(DATASETS) * len(MODELS_5CLASS)} models')

PLATT_THRESHOLD = 30
Datasets: ['nsl_kdd_v2', 'unsw_nb15_v2', 'cic_ids2017_v2']
Models to refit per dataset: ['rf_5class_smote', 'xgb_5class_smote', 'dnn_5class_smote', 'rf_5class_cw', 'xgb_5class_cw', 'dnn_5class_cw']
Total: 18 = 18 models


## 2. Calibrator fit/apply functions (copied verbatim from Notebook 03)

In [3]:
def fit_calibrator(p_calib, y_indicator, n_class):
    """Identical to Notebook 03 fit_calibrator."""
    if n_class >= PLATT_THRESHOLD:
        cal = IsotonicRegression(out_of_bounds='clip')
        cal.fit(p_calib, y_indicator)
        return cal, 'isotonic'
    else:
        cal = LogisticRegression(C=1e10, solver='lbfgs')
        cal.fit(p_calib.reshape(-1, 1), y_indicator)
        return cal, 'platt'

def apply_calibrator(calibrator, strategy, p_test):
    """Identical to Notebook 03 apply_calibrator."""
    if strategy == 'isotonic':
        return calibrator.predict(p_test)
    else:
        return calibrator.predict_proba(p_test.reshape(-1, 1))[:, 1]

print('✓ fit/apply functions loaded')

✓ fit/apply functions loaded


## 3. Path-resolution helper (handles NSL vs UNSW/CIC layout difference)

In [4]:
def find_proba_file(dataset, model_name, split):
    """Find a *_calib_proba.npy or *_test_proba.npy file.
    Checks both predictions/ (NSL) and probabilities/ (UNSW/CIC) folders.

    Args:
        dataset: e.g. 'nsl_kdd_v2'
        model_name: e.g. 'rf_5class_cw'
        split: 'calib' or 'test'
    """
    fname = f'{model_name}_{split}_proba.npy'
    for subdir in ['probabilities', 'predictions']:
        p = Path(REPO) / 'models' / dataset / subdir / fname
        if p.exists():
            return p
    raise FileNotFoundError(f'No {fname} found for {dataset}/{model_name} in predictions/ or probabilities/')

# Test path resolution on one example per dataset
for ds in DATASETS:
    p_calib = find_proba_file(ds, 'rf_5class_cw', 'calib')
    p_test = find_proba_file(ds, 'rf_5class_cw', 'test')
    print(f'{ds}: calib={p_calib.relative_to(REPO)}, test={p_test.relative_to(REPO)}')

nsl_kdd_v2: calib=models/nsl_kdd_v2/predictions/rf_5class_cw_calib_proba.npy, test=models/nsl_kdd_v2/predictions/rf_5class_cw_test_proba.npy
unsw_nb15_v2: calib=models/unsw_nb15_v2/probabilities/rf_5class_cw_calib_proba.npy, test=models/unsw_nb15_v2/probabilities/rf_5class_cw_test_proba.npy
cic_ids2017_v2: calib=models/cic_ids2017_v2/probabilities/rf_5class_cw_calib_proba.npy, test=models/cic_ids2017_v2/probabilities/rf_5class_cw_test_proba.npy


## 4. Refit and save calibrators for all 18 5-class models

In [ ]:
def refit_one_model(dataset, model_name):
    """Refit hybrid calibrators for one (dataset, model) and verify against saved output.
    Returns dict with calibrators, strategies, and verification status.
    """
    # Load labels
    y_calib = np.load(f'{REPO}/data/processed/{dataset}/y_calib_5class.npy')
    y_test = np.load(f'{REPO}/data/processed/{dataset}/y_test_5class.npy')

    # Load raw probabilities
    p_calib_2d = np.load(find_proba_file(dataset, model_name, 'calib'))
    p_test_2d = np.load(find_proba_file(dataset, model_name, 'test'))

    n_classes = p_calib_2d.shape[1]
    calib_counts = Counter(y_calib.tolist())

    # Refit per-class
    calibrators = {}
    strategies = {}
    p_test_cal = np.zeros_like(p_test_2d)

    for c in range(n_classes):
        y_calib_c = (y_calib == c).astype(int)
        p_calib_c = p_calib_2d[:, c]
        n_c = calib_counts.get(c, 0)

        cal, strat = fit_calibrator(p_calib_c, y_calib_c, n_c)
        calibrators[c] = cal
        strategies[c] = strat

        p_test_cal[:, c] = apply_calibrator(cal, strat, p_test_2d[:, c])

    # Renormalise to row sums == 1
    row_sums = p_test_cal.sum(axis=1, keepdims=True)
    row_sums = np.where(row_sums == 0, 1, row_sums)
    p_test_cal_renorm = p_test_cal / row_sums

    # VERIFY against saved output
    saved_path = Path(REPO) / 'calibrators' / dataset / f'{model_name}_test_proba_calibrated.npy'
    saved_calibrated = np.load(saved_path)

    matches = np.allclose(p_test_cal_renorm, saved_calibrated, atol=1e-10)
    max_diff = float(np.max(np.abs(p_test_cal_renorm - saved_calibrated)))

    return {
        'dataset': dataset,
        'model': model_name,
        'calibrators': calibrators,
        'strategies': strategies,
        'n_classes': n_classes,
        'calib_counts': dict(calib_counts),
        'verification_passed': bool(matches),
        'max_diff_vs_saved': max_diff,
    }

# Run refit on all 18 5-class models
print(f'\n{"="*70}')
print(f'REFIT VERIFICATION (atol=1e-10)')
print(f'{"="*70}\n')

all_results = []
all_passed = True

for ds in DATASETS:
    print(f'\n{ds}:')
    for model_name in MODELS_5CLASS:
        try:
            res = refit_one_model(ds, model_name)
            status = '✓ PASS' if res['verification_passed'] else '✗ FAIL'
            print(f'  {status:<8} {model_name:<22} max|diff|={res["max_diff_vs_saved"]:.2e}, strategies={res["strategies"]}')
            if not res['verification_passed']:
                all_passed = False
            all_results.append(res)
        except Exception as e:
            print(f'  ✗ ERROR {model_name:<22} {type(e).__name__}: {e}')
            all_passed = False

print(f'\n{"="*70}')
print(f'OVERALL: {"✓ ALL 18 MODELS VERIFIED" if all_passed else "✗ SOME FAILURES — DO NOT PROCEED"}')
print(f'{"="*70}')


REFIT VERIFICATION (atol=1e-10)


nsl_kdd_v2:
  ✓ PASS   rf_5class_smote        max|diff|=0.00e+00, strategies={0: 'isotonic', 1: 'isotonic', 2: 'isotonic', 3: 'isotonic', 4: 'platt'}
  ✓ PASS   xgb_5class_smote       max|diff|=0.00e+00, strategies={0: 'isotonic', 1: 'isotonic', 2: 'isotonic', 3: 'isotonic', 4: 'platt'}
  ✓ PASS   dnn_5class_smote       max|diff|=0.00e+00, strategies={0: 'isotonic', 1: 'isotonic', 2: 'isotonic', 3: 'isotonic', 4: 'platt'}
  ✓ PASS   rf_5class_cw           max|diff|=0.00e+00, strategies={0: 'isotonic', 1: 'isotonic', 2: 'isotonic', 3: 'isotonic', 4: 'platt'}
  ✓ PASS   xgb_5class_cw          max|diff|=0.00e+00, strategies={0: 'isotonic', 1: 'isotonic', 2: 'isotonic', 3: 'isotonic', 4: 'platt'}
  ✓ PASS   dnn_5class_cw          max|diff|=0.00e+00, strategies={0: 'isotonic', 1: 'isotonic', 2: 'isotonic', 3: 'isotonic', 4: 'platt'}

unsw_nb15_v2:


## 5. Save fitted calibrators (only if all verified)

In [ ]:
if not all_passed:
    print('⚠ Some models failed verification. Skipping save.')
    print('Inspect the failures above before proceeding.')
else:
    print('All 18 models verified. Saving fitted calibrators...')
    saved_count = 0
    for res in all_results:
        out_dir = Path(REPO) / 'calibrators' / res['dataset']
        out_dir.mkdir(parents=True, exist_ok=True)
        out_path = out_dir / f'{res["model"]}_hybrid_fitted.joblib'

        to_save = {
            'calibrators': res['calibrators'],
            'strategies': res['strategies'],
            'n_classes': res['n_classes'],
            'calib_counts': res['calib_counts'],
            'platt_threshold': PLATT_THRESHOLD,
        }
        joblib.dump(to_save, out_path)
        saved_count += 1
        print(f'  Saved: {out_path.relative_to(REPO)}')
    print(f'\n✓ Saved {saved_count} fitted calibrator bundles')

## 6. Round-trip sanity check (load saved + apply)

In [ ]:
if all_passed:
    # Load one saved calibrator bundle and verify it produces the right output
    test_ds = 'nsl_kdd_v2'
    test_model = 'dnn_5class_cw'

    bundle_path = Path(REPO) / 'calibrators' / test_ds / f'{test_model}_hybrid_fitted.joblib'
    bundle = joblib.load(bundle_path)

    # Load raw test probs and labels
    p_test_2d = np.load(find_proba_file(test_ds, test_model, 'test'))

    # Apply loaded calibrators
    p_test_cal_roundtrip = np.zeros_like(p_test_2d)
    for c in range(bundle['n_classes']):
        p_test_cal_roundtrip[:, c] = apply_calibrator(
            bundle['calibrators'][c], bundle['strategies'][c], p_test_2d[:, c]
        )

    # Renormalise
    row_sums = p_test_cal_roundtrip.sum(axis=1, keepdims=True)
    row_sums = np.where(row_sums == 0, 1, row_sums)
    p_test_cal_roundtrip = p_test_cal_roundtrip / row_sums

    # Compare against saved original calibrated output
    saved = np.load(f'{REPO}/calibrators/{test_ds}/{test_model}_test_proba_calibrated.npy')
    max_diff = float(np.max(np.abs(p_test_cal_roundtrip - saved)))

    print(f'Round-trip test on {test_ds}/{test_model}:')
    print(f'  max |diff| = {max_diff:.2e}')
    print(f'  Status: {"✓ ROUND-TRIP PASS" if max_diff < 1e-10 else "✗ ROUND-TRIP FAIL"}')

    print(f'\n✓ Refit complete. Ready for Notebook 04b validation experiment.')
else:
    print('Skipping round-trip test (verification failed earlier).')

## 7. Commit

In [ ]:
if all_passed:
    os.chdir(REPO)
    !git add notebooks/03e_refit_hybrid_calibrators.ipynb
    !git add calibrators/
    !git status --short
    !git commit -m 'Notebook 03e: refit hybrid calibrators as joblib objects (verified <1e-10 vs saved probs)'
    !git push origin main
else:
    print('Verification failed — not committing. Fix issues and re-run.')